In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix


In [2]:
import glob
import os

base_dir = "/data1st1/junyi/correctdata/GSE133549/*/"
human_files = sorted(glob.glob(os.path.join(base_dir, "*human*exp_mat*.tsv.gz")))
print(f"Found {len(human_files)} human matrix file(s):")
for f in human_files:
    print(" -", os.path.basename(f))

file_path = human_files[0]

Found 14 human matrix file(s):
 - GSE133535_10X2x5Kcell250Kreads_human_exp_mat.tsv.gz
 - GSE133536_10X8x10Kcell25Kreads_human_exp_mat.tsv.gz
 - GSE133537_C1HTmedium_human_exp_mat.tsv.gz
 - GSE133538_C1HTsmall_human_exp_mat.tsv.gz
 - GSE133539_CELseq2_human_exp_mat.tsv.gz
 - GSE133540_Dropseq_human_exp_mat.tsv.gz
 - GSE133541_ICELL8_human_exp_mat.tsv.gz
 - GSE133542_MARSseq_human_exp_mat.tsv.gz
 - GSE133543_QUARTZseq_human_exp_mat.tsv.gz
 - GSE133544_SCRBseq_human_exp_mat.tsv.gz
 - GSE133545_SMARTseq2_human_exp_mat.tsv.gz
 - GSE133546_SingleNuclei_human_exp_mat.tsv.gz
 - GSE133547_ddSEQ_human_exp_mat.tsv.gz
 - GSE133548_inDrop_human_exp_mat.tsv.gz


In [3]:
# Load each human matrix, stamp its filename into obs, then concatenate
adatas = []
for fp in human_files:
    expr_matrix = pd.read_csv(fp, sep='\t', index_col=0, compression='gzip')
    mat_X = csr_matrix(np.round(expr_matrix.T.values).astype(int))

    ad = sc.AnnData(
        X=mat_X,
        obs=pd.DataFrame(index=expr_matrix.columns),
        var=pd.DataFrame(index=expr_matrix.index),
    )
    ad.obs["source_file"] = os.path.basename(fp)
    ad.var_names_make_unique()
    adatas.append(ad)

adata = sc.concat(adatas, join="outer", label="sample_id", keys=[os.path.basename(f) for f in human_files])

print(adata)
print(adata.obs.head())

AnnData object with n_obs × n_vars = 139617 × 47768
    obs: 'source_file', 'sample_id'
    layers: None (.X)
                                                                      source_file  \
10X2x5K_64220_AAACCTGAGACGCACA  GSE133535_10X2x5Kcell250Kreads_human_exp_mat.t...   
10X2x5K_64220_AAACCTGAGCCACCTG  GSE133535_10X2x5Kcell250Kreads_human_exp_mat.t...   
10X2x5K_64220_AAACCTGAGTGCGTGA  GSE133535_10X2x5Kcell250Kreads_human_exp_mat.t...   
10X2x5K_64220_AAACCTGCACTGTGTA  GSE133535_10X2x5Kcell250Kreads_human_exp_mat.t...   
10X2x5K_64220_AAACCTGCAGACTCGC  GSE133535_10X2x5Kcell250Kreads_human_exp_mat.t...   

                                                                        sample_id  
10X2x5K_64220_AAACCTGAGACGCACA  GSE133535_10X2x5Kcell250Kreads_human_exp_mat.t...  
10X2x5K_64220_AAACCTGAGCCACCTG  GSE133535_10X2x5Kcell250Kreads_human_exp_mat.t...  
10X2x5K_64220_AAACCTGAGTGCGTGA  GSE133535_10X2x5Kcell250Kreads_human_exp_mat.t...  
10X2x5K_64220_AAACCTGCACTGTGTA  GSE133535_1

In [4]:
adata.X

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 232968948 stored elements and shape (139617, 47768)>

In [5]:
adata.write_h5ad("/data1st1/junyi/correctdata/GSE133549/human_exp_mat.h5ad")